# THE TRAINING & TESTING DATA IS READY, RUN IN CAUTIONS
# THIS IS ONLY USED TO PROCESS AMAZON-2014 JSON DATA

In [ ]:
import torch
torch.__version__
import pandas as pd
pd.set_option('display.max_colwidth', None)
from sklearn import preprocessing as pp
from sklearn.model_selection import train_test_split
import numpy as np
import os
from pathlib import Path
from variable import *

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

Data Preprocessing 1, map user & item json id to number only index (start from 0)

In [ ]:
def save_mapping(sfile):
    df = pd.read_json(sfile, lines=True)
    df = df[['reviewerID', 'asin', 'overall']]
    df.columns = ['user_id', 'item_id', 'rating']
    print(len(df))
    display(df.head(5))
    # Check the number of interactions per user and per item
    user_counts = df['user_id'].value_counts()
    item_counts = df['item_id'].value_counts()
    print("Before filtering:")
    print(f"Total users: {len(user_counts)}")
    print(f"Total items: {len(item_counts)}")
    # Keep users with at least 5 interactions
    df = df[df['user_id'].isin(user_counts[user_counts >= 5].index)]
    # Keep items with at least 5 interactions
    df = df[df['item_id'].isin(item_counts[item_counts >= 5].index)]
    # Recalculate stats after filtering
    user_counts_after = df['user_id'].value_counts()
    item_counts_after = df['item_id'].value_counts()
    print("\nAfter filtering:")
    print(f"Total users: {len(user_counts_after)}")
    print(f"Total items: {len(item_counts_after)}")
    print(f"Total interactions: {len(df)}")
    # user & item mapping
    le_user = pp.LabelEncoder()
    le_item = pp.LabelEncoder()
    df['user_id_idx'] = le_user.fit_transform(df['user_id'].values)
    df['item_id_idx'] = le_item.fit_transform(df['item_id'].values)
    n_users = df['user_id_idx'].nunique()
    n_items = df['item_id_idx'].nunique()
    print("Number of Unique Users : ", n_users)
    print("Number of unique Items : ", n_items)
    # map dict, txt file
    le_user_mapping = dict(zip(le_user.classes_, le_user.transform(le_user.classes_)))
    le_item_mapping = dict(zip(le_item.classes_, le_item.transform(le_item.classes_)))
    with open(str(Path(sfile).parent) + "user-map.txt", "w") as output:
        for k, v in le_user_mapping.items():
            output.write(f'{k} {v}\n')
    with open(str(Path(sfile).parent) + "item-map.txt", "w") as output:
        for k, v in le_item_mapping.items():
            output.write(f'{k} {v}\n')
    return user_counts_after, item_counts_after, df

In [ ]:
# Example Usage of mapping JSON id to ID 
# music_user_count, music_item_count, music_dataframe = save_mapping(amazon_music)

Data Preprocess 2, create CDR overlap user-id txt

In [ ]:
mappings = {}
for domain, path in user_map_paths.items():
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["original_id", "mapped_id"])
    mappings[domain] = df
    print(f"Loaded {domain}: {len(df)} users")
# Function to find overlap between two domains
def save_overlap(domain_a, domain_b):
    df_a = mappings[domain_a]
    df_b = mappings[domain_b]
    # Inner join on original_id to find overlap
    overlap = pd.merge(df_a, df_b, on="original_id", how="inner", suffixes=(f"_{domain_a}", f"_{domain_b}"))
    # Select only mapped IDs for output
    out_df = overlap[[f"mapped_id_s", f"mapped_id_t"]]
    # Save
    out_path = os.path.join(overlap_save, f"{domain_a}-{domain_b}-overlap.txt")
    out_df.to_csv(out_path, sep=' ', index=False, header=False)
    print(f"✅ Saved {len(out_df)} overlaps -> {out_path}")
    return out_path, out_df

In [ ]:
# Example Usage of Generate all three overlap files
# book-movie-overlap, book-movie-dataframe = save_overlap("book", "movie")

Data Preprocess 3, split and save the ratings data for CDR

In [ ]:
def split_and_save(ratio, overlap_df, target_df, total_n_users, total_n_items, total_tn_users, total_tn_items, save_dir):
    """
    Splits overlap users into train/test sets based on a given ratio,
    and saves the resulting interactions and metadata.

    Args:
        ratio (float): The ratio of users to be used for the train set (e.g., 0.2 or 0.5).
        overlap_df (pd.DataFrame): DataFrame of mapping for overlap users from two domainz.
        target_df (pd.DataFrame): DataFrame of interactions for target domain.
        total_n_users (int): Total number of unique users in the music domain.
        total_n_items (int): Total number of unique items in the music domain.
        save_dir (str): Directory to save the output files.
    """

    # Define the path to the overlap file
    overlap_path = os.path.join(overlap_save, "book-movie-overlap.txt")
    print(f"Loading overlap users from: {overlap_path}")
    # Load the overlap users
    # The file contains: mapped_id_movie, mapped_id_music (which is the user_id_idx in df)
    overlap_df = pd.read_csv(overlap_path, sep=' ', header=None, names=["mapped_id_s", "mapped_id_t"])
    # The 'mapped_id_music' column contains the 'user_id_idx' for the Music domain.
    overlap_user_indices = overlap_df['mapped_id_t'].unique()
    print(f"Total number of overlap users: {len(overlap_user_indices)}")
    # Filter the original music DataFrame (df) to include only interactions from overlap users
    overlap_df_interactions = target_df[target_df['user_id_idx'].isin(overlap_user_indices)].copy()
    print(f"Total interactions for overlap users: {len(overlap_df_interactions)}")
    
    # 1. Split the overlap users into train/test
    all_overlap_users = overlap_df_interactions['user_id_idx'].unique()
    
    # train_users will be 'ratio' percent of the overlap users, test_users will be the rest.
    train_users, test_users = train_test_split(
        all_overlap_users, 
        test_size=(1 - ratio), 
        random_state=42  # For reproducibility
    )
    
    print(f"\n--- Splitting with ratio {ratio*100:.0f}% ---")
    print(f"Train users ({ratio*100:.0f}%): {len(train_users)}")
    print(f"Test users ({1-ratio:.0f}%): {len(test_users)}")

    # 2. Split interactions based on user sets
    train_df = overlap_df_interactions[overlap_df_interactions['user_id_idx'].isin(train_users)]
    test_df = overlap_df_interactions[overlap_df_interactions['user_id_idx'].isin(test_users)]
    
    print(f"Train interactions: {len(train_df)}")
    print(f"Test interactions: {len(test_df)}")

    # 3. Save the train DataFrame as a .txt file (user_id, item_id, rating)
    txt_filename = f"{int(ratio*100)}train.txt"
    txt_path = os.path.join(save_dir, txt_filename)
    # Using the mapped ID columns (user_id_idx, item_id_idx) and rating
    train_df[['user_id_idx', 'item_id_idx', 'rating']].to_csv(
        txt_path, 
        sep=' ', 
        index=False, 
        header=False
    )
    print(f"✅ Saved training interactions to: {txt_path}")

    # 4. Prepare data for .pt file (PyTorch tensor)
    # Variables required: train_uid, train_iid, train_rates, test_uid, test_iid, test_rates, n_user, n_item
    
    # Convert dataframes to numpy arrays of the correct type (e.g., int/long)
    train_uid = train_df['user_id_idx'].values.astype(np.int64)
    train_iid = train_df['item_id_idx'].values.astype(np.int64)
    train_rates = train_df['rating'].values.astype(np.float32)

    test_uid = test_df['user_id_idx'].values.astype(np.int64)
    test_iid = test_df['item_id_idx'].values.astype(np.int64)
    test_rates = test_df['rating'].values.astype(np.float32)

    # Convert numpy arrays to PyTorch tensors
    data_to_save = {
        'train_uid': torch.from_numpy(train_uid),
        'train_iid': torch.from_numpy(train_iid),
        'train_rates': torch.from_numpy(train_rates),
        'test_uid': torch.from_numpy(test_uid),
        'test_iid': torch.from_numpy(test_iid),
        'test_rates': torch.from_numpy(test_rates),
        # Total number of unique users/items in the *entire* music domain (not just overlap)
        'n_user': total_n_users, 
        'n_item': total_n_items,
        'tn_user': total_tn_users, 
        'tn_item': total_tn_items
    }

    # Save the PyTorch data
    pt_filename = f"{int(ratio*100)}train.pt"
    pt_path = os.path.join(save_dir, pt_filename)
    torch.save(data_to_save, pt_path)
    print(f"✅ Saved PyTorch data to: {pt_path}")

Loading overlap users from: /remote-home/share/dmb_nas2/yii/Data/amazon-2014/book-movie-overlap.txt
Total number of overlap users: 37388
Total interactions for overlap users: 792319

--- Splitting with ratio 20% ---
Train users (20%): 7477
Test users (1%): 29911
Train interactions: 158437
Test interactions: 633882
✅ Saved training interactions to: /remote-home/share/dmb_nas2/yii/Data/amazon-cold/book-movie/20train.txt
✅ Saved PyTorch data to: /remote-home/share/dmb_nas2/yii/Data/amazon-cold/book-movie/20train.pt

--- Splitting with ratio 50% ---
Train users (50%): 18694
Test users (0%): 18694
Train interactions: 395050
Test interactions: 397269
✅ Saved training interactions to: /remote-home/share/dmb_nas2/yii/Data/amazon-cold/book-movie/50train.txt
✅ Saved PyTorch data to: /remote-home/share/dmb_nas2/yii/Data/amazon-cold/book-movie/50train.pt


In [ ]:
# example usage
# **1. 20% Train / 80% Test Split**

# split_and_save(
#     ratio=0.2, 
#     overlap_df=book-movie-dataframe,
#     target_df=movie_dataframe, 
#     total_n_users=book_user_count, 
#     total_n_items=book_item_count, 
#     total_tn_users=movie_user_count, 
#     total_tn_items=movie_item_count,
#     save_dir=book_movie_save
# )